In [1]:
!pip install torch torch-geometric scikit-learn -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 743.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.7 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()  # select all 8 JSON files
DATA_DIR = '/content'


Saving a_writes.json to a_writes (1).json
Saving advisors.json to advisors (1).json
Saving belongs_to.json to belongs_to (1).json
Saving courses.json to courses (1).json
Saving experts_in.json to experts_in (1).json
Saving interested_in.json to interested_in (1).json
Saving papers.json to papers (1).json
Saving research_area.json to research_area (1).json
Saving students.json to students (1).json
Saving takes.json to takes (1).json


In [3]:
import json
import os
import sys
import torch
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from torch_geometric.data import HeteroData

In [4]:
# Fix Windows console encoding
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# ── Path Configuration ──────────────────────────────────────────────────────
DATA_DIR = os.path.dirname(os.path.abspath(__file__))


NameError: name '__file__' is not defined

In [5]:
# ═══════════════════════════════════════════════════════════════════════════
#  1. DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════
def load_json(filename):
    """Load a JSON file from the data directory."""
    filepath = os.path.join(DATA_DIR, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def load_all_data():
    """Load all JSON datasets and return as a dictionary."""
    print("[*] Loading JSON datasets...")
    data = {
        "advisors":      load_json("advisors.json"),
        "students":      load_json("students.json"),
        "papers":        load_json("papers.json"),
        "courses":       load_json("courses.json"),
        "research_areas": load_json("research_area.json"),
        "a_writes":      load_json("a_writes.json"),
        "belongs_to":    load_json("belongs_to.json"),
        "experts_in":    load_json("experts_in.json"),
        "interested_in": load_json("interested_in.json"),
        "takes":         load_json("takes.json"),
    }

    # Filter out null/empty entries
    data["students"] = [s for s in data["students"] if s.get("student_id") is not None]
    data["advisors"] = [a for a in data["advisors"] if a.get("name") is not None]

    print(f"   ✓ Advisors:       {len(data['advisors'])}")
    print(f"   ✓ Students:       {len(data['students'])}")
    print(f"   ✓ Papers:         {len(data['papers'])}")
    print(f"   ✓ Courses:        {len(data['courses'])}")
    print(f"   ✓ Research Areas: {len(data['research_areas'])}")
    print(f"   ✓ Writes edges:   {len(data['a_writes'])}")
    print(f"   ✓ Belongs_to:     {len(data['belongs_to'])}")
    print(f"   ✓ Experts_in:     {len(data['experts_in'])}")
    print(f"   ✓ Interested_in:  {len(data['interested_in'])}")
    print(f"   ✓ Takes edges:    {len(data['takes'])}")

    return data



In [6]:
# ═══════════════════════════════════════════════════════════════════════════
#  2. INDEX MAPPING
# ═══════════════════════════════════════════════════════════════════════════
def build_node_mappings(data):
    """
    Create integer index mappings for every node type.
    Returns: dict of {node_type: {original_id: int_index}}
    """
    print("\n[*] Building node index mappings...")

    mappings = {}

    # Advisor: name → index
    mappings["advisor"] = {
        a["name"]: i for i, a in enumerate(data["advisors"])
    }

    # Student: student_id → index
    mappings["student"] = {
        s["student_id"]: i for i, s in enumerate(data["students"])
    }

    # Paper: paper_index → index (0-based)
    mappings["paper"] = {
        p["paper_index"]: i for i, p in enumerate(data["papers"])
    }

    # Course: course_id → index
    mappings["course"] = {
        c["course_id"]: i for i, c in enumerate(data["courses"])
    }

    # Research Area: id → index (0-based)
    mappings["research_area"] = {
        r["id"]: i for i, r in enumerate(data["research_areas"])
    }

    for ntype, m in mappings.items():
        print(f"   ✓ {ntype:20s} → {len(m):>4d} nodes")

    return mappings



In [7]:
# ═══════════════════════════════════════════════════════════════════════════
#  3. NODE FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════
def collect_global_vocabulary(data):
    """
    Build a global vocabulary of all unique topic / research-interest strings
    across advisors, students, and research areas for multi-hot encoding.
    """
    vocab = set()
    for a in data["advisors"]:
        vocab.update(a.get("publication_topics", []))
        vocab.update(a.get("research_areas", []))
    for s in data["students"]:
        vocab.update(s.get("research_interests", []))
    for r in data["research_areas"]:
        vocab.add(r["research_area"])
    vocab.discard("")
    vocab = sorted(vocab)
    return {term: i for i, term in enumerate(vocab)}


def multi_hot(terms, vocab_map, dim):
    """Create a multi-hot vector from a list of terms."""
    vec = np.zeros(dim, dtype=np.float32)
    for t in terms:
        if t in vocab_map:
            vec[vocab_map[t]] = 1.0
    return vec


def build_advisor_features(data, vocab_map, vocab_dim):
    """
    Advisor features:
      [designation_onehot | pub_count_norm | capacity_norm | multi_hot(topics+areas)]
    """
    designations = sorted(set(a["designation"] for a in data["advisors"]))
    desig_map = {d: i for i, d in enumerate(designations)}
    desig_dim = len(designations)

    features = []
    for a in data["advisors"]:
        # Designation one-hot
        desig_vec = np.zeros(desig_dim, dtype=np.float32)
        desig_vec[desig_map[a["designation"]]] = 1.0

        # Numerical features (normalised later)
        pub_count = float(a.get("publication_count", 0) or 0)
        capacity = float(a.get("capacity", 0) or 0)

        # Multi-hot topic + area embedding
        topics = a.get("publication_topics", [])
        areas = a.get("research_areas", [])
        mh = multi_hot(topics + areas, vocab_map, vocab_dim)

        feat = np.concatenate([desig_vec, [pub_count, capacity], mh])
        features.append(feat)

    features = np.stack(features)

    # Normalise numerical columns (pub_count, capacity)
    num_start = desig_dim
    num_end = desig_dim + 2
    scaler = MinMaxScaler()
    features[:, num_start:num_end] = scaler.fit_transform(
        features[:, num_start:num_end]
    )

    return torch.tensor(features, dtype=torch.float)


def build_student_features(data, vocab_map, vocab_dim, course_map):
    """
    Student features:
      [cgpa_norm | multi_hot(completed_courses) | multi_hot(research_interests)]
    """
    course_dim = len(course_map)
    features = []

    for s in data["students"]:
        cgpa = float(s.get("cgpa", 0) or 0) / 4.0  # normalise to [0, 1]

        # Completed courses multi-hot
        course_vec = np.zeros(course_dim, dtype=np.float32)
        for cid in s.get("completed_courses", []):
            if cid in course_map:
                course_vec[course_map[cid]] = 1.0

        # Research interests multi-hot
        interests = s.get("research_interests", [])
        interest_vec = multi_hot(interests, vocab_map, vocab_dim)

        feat = np.concatenate([[cgpa], course_vec, interest_vec])
        features.append(feat)

    features = np.stack(features)
    return torch.tensor(features, dtype=torch.float)


def build_paper_features(data):
    """
    Paper features:
      TF-IDF embedding of the paper title (dimensionality-reduced).
    """
    titles = [p["paper_title"] for p in data["papers"]]

    tfidf = TfidfVectorizer(
        max_features=128,      # compact embedding dimension
        stop_words="english",
        sublinear_tf=True,
    )
    X = tfidf.fit_transform(titles).toarray().astype(np.float32)
    return torch.tensor(X, dtype=torch.float)


def build_course_features(data, vocab_map, vocab_dim):
    """
    Course features:
      [one_hot(course_id) | TF-IDF of course name]
    """
    n_courses = len(data["courses"])
    names = [c["course_name"] for c in data["courses"]]

    tfidf = TfidfVectorizer(max_features=32, stop_words="english")
    X_text = tfidf.fit_transform(names).toarray().astype(np.float32)

    identity = np.eye(n_courses, dtype=np.float32)
    features = np.concatenate([identity, X_text], axis=1)
    return torch.tensor(features, dtype=torch.float)


def build_research_area_features(data, vocab_map, vocab_dim):
    """
    Research Area features:
      [one_hot(area_id) | multi_hot from global vocab]
    """
    n_areas = len(data["research_areas"])
    features = []

    for i, r in enumerate(data["research_areas"]):
        identity = np.zeros(n_areas, dtype=np.float32)
        identity[i] = 1.0

        mh = multi_hot([r["research_area"]], vocab_map, vocab_dim)
        feat = np.concatenate([identity, mh])
        features.append(feat)

    features = np.stack(features)
    return torch.tensor(features, dtype=torch.float)



In [8]:
# ═══════════════════════════════════════════════════════════════════════════
#  4. EDGE INDEX CONSTRUCTION
# ═══════════════════════════════════════════════════════════════════════════
def build_edge_indices(data, mappings):
    """
    Construct edge_index tensors for each relation type.
    Returns dict: {(src_type, rel, dst_type): edge_index_tensor}
    """
    print("\n[*] Building edge indices...")
    edges = {}

    # ── advisor WRITES paper ─────────────────────────────────────────────
    src, dst = [], []
    for e in data["a_writes"]:
        advisor_name = e.get("advisor")
        # Handle the typo: some entries have "-paper_index" instead of "paper_index"
        paper_idx = e.get("paper_index") or e.get("-paper_index")
        if advisor_name in mappings["advisor"] and paper_idx in mappings["paper"]:
            src.append(mappings["advisor"][advisor_name])
            dst.append(mappings["paper"][paper_idx])
    edges[("advisor", "writes", "paper")] = torch.tensor([src, dst], dtype=torch.long)
    print(f"   ✓ advisor  ──writes──▸  paper        : {len(src):>4d} edges")

    # ── paper BELONGS_TO research_area ───────────────────────────────────
    src, dst = [], []
    for e in data["belongs_to"]:
        pid = e.get("paper_index")
        raid = e.get("research_area_id")
        if pid in mappings["paper"] and raid in mappings["research_area"]:
            src.append(mappings["paper"][pid])
            dst.append(mappings["research_area"][raid])
    edges[("paper", "belongs_to", "research_area")] = torch.tensor([src, dst], dtype=torch.long)
    print(f"   ✓ paper    ──belongs──▸ research_area: {len(src):>4d} edges")

    # ── advisor EXPERT_IN research_area ──────────────────────────────────
    src, dst = [], []
    for e in data["experts_in"]:
        advisor_name = e.get("advisor")
        area = e.get("research_area")
        # Some entries have area as [] (empty list) — skip those
        if isinstance(area, list):
            continue
        # Map area name → research_area id
        area_id = None
        for r in data["research_areas"]:
            if r["research_area"] == area:
                area_id = r["id"]
                break
        if advisor_name in mappings["advisor"] and area_id in mappings["research_area"]:
            src.append(mappings["advisor"][advisor_name])
            dst.append(mappings["research_area"][area_id])
    edges[("advisor", "expert_in", "research_area")] = torch.tensor([src, dst], dtype=torch.long)
    print(f"   ✓ advisor  ──expert──▸  research_area: {len(src):>4d} edges")

    # ── student INTERESTED_IN research_area ──────────────────────────────
    src, dst = [], []
    for e in data["interested_in"]:
        sid = e.get("student_id")
        area = e.get("research_area")
        area_id = None
        for r in data["research_areas"]:
            if r["research_area"] == area:
                area_id = r["id"]
                break
        if sid in mappings["student"] and area_id is not None and area_id in mappings["research_area"]:
            src.append(mappings["student"][sid])
            dst.append(mappings["research_area"][area_id])
    edges[("student", "interested_in", "research_area")] = torch.tensor([src, dst], dtype=torch.long)
    print(f"   ✓ student  ──interest──▸research_area: {len(src):>4d} edges")

    # ── student TAKES course ─────────────────────────────────────────────
    src, dst = [], []
    for e in data["takes"]:
        sid = e.get("student_id")
        cid = e.get("course_id")
        if sid in mappings["student"] and cid in mappings["course"]:
            src.append(mappings["student"][sid])
            dst.append(mappings["course"][cid])
    edges[("student", "takes", "course")] = torch.tensor([src, dst], dtype=torch.long)
    print(f"   ✓ student  ──takes──▸   course       : {len(src):>4d} edges")

    return edges



In [9]:
# ═══════════════════════════════════════════════════════════════════════════
#  5. HETEROGENEOUS GRAPH ASSEMBLY
# ═══════════════════════════════════════════════════════════════════════════
def build_hetero_data(node_features, edge_indices):
    """
    Assemble a PyG HeteroData object from node features and edge indices.
    Also adds reverse edges for message passing.
    """
    print("\n[*] Assembling HeteroData graph...")
    hdata = HeteroData()

    # ── Attach node features ─────────────────────────────────────────────
    for ntype, feat in node_features.items():
        hdata[ntype].x = feat
        hdata[ntype].num_nodes = feat.size(0)
        print(f"   Node [{ntype:15s}]  shape: {list(feat.shape)}")

    # ── Attach edge indices + add reverse edges ──────────────────────────
    for (src_type, rel, dst_type), ei in edge_indices.items():
        hdata[src_type, rel, dst_type].edge_index = ei
        # Add reverse edge for bidirectional message passing
        rev_rel = f"rev_{rel}"
        hdata[dst_type, rev_rel, src_type].edge_index = ei.flip(0)
        print(f"   Edge ({src_type}, {rel}, {dst_type})  "
              f"→  {ei.size(1)} edges  (+reverse)")

    return hdata



In [10]:
# ═══════════════════════════════════════════════════════════════════════════
#  6. HOMOGENEOUS PROJECTION (for simple GCN)
# ═══════════════════════════════════════════════════════════════════════════
def project_to_homogeneous(hdata, target_dim=128):
    """
    Project the heterogeneous graph to a homogeneous graph by:
      1. Linearly projecting all node features to a shared dimension
      2. Merging all nodes & edges with global index offsets

    Returns:
      - homo_x:          (N_total, target_dim) float tensor
      - homo_edge_index: (2, E_total) long tensor
      - node_type_labels: (N_total,) int tensor  — for node classification
      - node_offsets:     dict {ntype: offset}
    """
    from torch import nn

    print(f"\n[*] Projecting to homogeneous graph (dim={target_dim})...")

    node_types = list(hdata.node_types)
    node_offsets = {}
    projected_features = []
    type_labels = []

    offset = 0
    for i, ntype in enumerate(node_types):
        feat = hdata[ntype].x
        n_nodes = feat.size(0)
        in_dim = feat.size(1)

        # Linear projection to shared space
        projector = nn.Linear(in_dim, target_dim, bias=False)
        nn.init.xavier_uniform_(projector.weight)
        with torch.no_grad():
            proj_feat = projector(feat)

        projected_features.append(proj_feat)
        type_labels.extend([i] * n_nodes)
        node_offsets[ntype] = offset
        offset += n_nodes
        print(f"   [{ntype:15s}]  {in_dim:>4d}d → {target_dim}d  "
              f"(offset: {node_offsets[ntype]})")

    homo_x = torch.cat(projected_features, dim=0)
    type_labels = torch.tensor(type_labels, dtype=torch.long)

    # Merge edges with global offsets
    all_src, all_dst = [], []
    for (src_type, rel, dst_type) in hdata.edge_types:
        ei = hdata[src_type, rel, dst_type].edge_index
        src_offset = node_offsets[src_type]
        dst_offset = node_offsets[dst_type]
        all_src.append(ei[0] + src_offset)
        all_dst.append(ei[1] + dst_offset)

    homo_edge_index = torch.stack([
        torch.cat(all_src),
        torch.cat(all_dst),
    ], dim=0)

    print(f"\n   Total nodes: {homo_x.size(0)}")
    print(f"   Total edges: {homo_edge_index.size(1)}")
    print(f"   Node types:  {dict(zip(node_types, range(len(node_types))))}")

    return homo_x, homo_edge_index, type_labels, node_offsets



In [11]:
# ═══════════════════════════════════════════════════════════════════════════
#  7. MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════
def main():
    print("=" * 72)
    print("  Academic Knowledge Graph — Dataset Embedding Pipeline")
    print("=" * 72)

    # Step 1: Load raw data
    data = load_all_data()

    # Step 2: Build index mappings
    mappings = build_node_mappings(data)

    # Step 3: Build global vocabulary for multi-hot encoding
    vocab_map = collect_global_vocabulary(data)
    vocab_dim = len(vocab_map)
    print(f"\n[*] Global vocabulary size: {vocab_dim} terms")

    # Step 4: Build course map for student features
    course_map = {c["course_id"]: i for i, c in enumerate(data["courses"])}

    # Step 5: Compute node features for each type
    print("\n[*] Engineering node features...")
    node_features = {
        "advisor":       build_advisor_features(data, vocab_map, vocab_dim),
        "student":       build_student_features(data, vocab_map, vocab_dim, course_map),
        "paper":         build_paper_features(data),
        "course":        build_course_features(data, vocab_map, vocab_dim),
        "research_area": build_research_area_features(data, vocab_map, vocab_dim),
    }
    for ntype, feat in node_features.items():
        print(f"   ✓ {ntype:15s}  →  {list(feat.shape)}")

    # Step 6: Build edge indices
    edge_indices = build_edge_indices(data, mappings)

    # Step 7: Assemble heterogeneous graph
    hdata = build_hetero_data(node_features, edge_indices)

    # Step 8: Create homogeneous projection
    homo_x, homo_edge_index, node_type_labels, node_offsets = \
        project_to_homogeneous(hdata, target_dim=128)

    # Step 9: Save everything
    print("\n[*] Saving processed graph data...")

    save_dir = os.path.join(DATA_DIR, "processed")
    os.makedirs(save_dir, exist_ok=True)

    # Save heterogeneous graph
    hetero_path = os.path.join(save_dir, "academic_graph_hetero.pt")
    torch.save(hdata, hetero_path)
    print(f"   ✓ HeteroData       → {hetero_path}")

    # Save homogeneous projection
    homo_path = os.path.join(save_dir, "academic_graph_homo.pt")
    torch.save({
        "x": homo_x,
        "edge_index": homo_edge_index,
        "node_type_labels": node_type_labels,
        "node_offsets": node_offsets,
    }, homo_path)
    print(f"   ✓ Homogeneous data → {homo_path}")

    # Save mappings
    mappings_path = os.path.join(save_dir, "node_mappings.pt")
    torch.save(mappings, mappings_path)
    print(f"   ✓ Node mappings    → {mappings_path}")

    # ── Summary ──────────────────────────────────────────────────────────
    print("\n" + "=" * 72)
    print("  ✅ EMBEDDING COMPLETE — Graph Summary")
    print("=" * 72)
    print(f"  {'Node Type':<20s} {'Nodes':>6s}   {'Feature Dim':>11s}")
    print(f"  {'─'*20} {'─'*6}   {'─'*11}")
    for ntype, feat in node_features.items():
        print(f"  {ntype:<20s} {feat.size(0):>6d}   {feat.size(1):>11d}")

    total_edges = sum(ei.size(1) for ei in edge_indices.values())
    print(f"\n  Total edges (directed):     {total_edges}")
    print(f"  Homogeneous projection dim: {homo_x.size(1)}")
    print(f"  Homogeneous total nodes:    {homo_x.size(0)}")
    print(f"  Homogeneous total edges:    {homo_edge_index.size(1)}")
    print("=" * 72)

    return hdata, homo_x, homo_edge_index, node_type_labels, node_offsets


if __name__ == "__main__":
    main()


  Academic Knowledge Graph — Dataset Embedding Pipeline
[*] Loading JSON datasets...
   ✓ Advisors:       25
   ✓ Students:       2
   ✓ Papers:         326
   ✓ Courses:        10
   ✓ Research Areas: 44
   ✓ Writes edges:   326
   ✓ Belongs_to:     334
   ✓ Experts_in:     115
   ✓ Interested_in:  11
   ✓ Takes edges:    6

[*] Building node index mappings...
   ✓ advisor              →   25 nodes
   ✓ student              →    2 nodes
   ✓ paper                →  326 nodes
   ✓ course               →   10 nodes
   ✓ research_area        →   44 nodes

[*] Global vocabulary size: 99 terms

[*] Engineering node features...
   ✓ advisor          →  [25, 105]
   ✓ student          →  [2, 110]
   ✓ paper            →  [326, 128]
   ✓ course           →  [10, 39]
   ✓ research_area    →  [44, 143]

[*] Building edge indices...
   ✓ advisor  ──writes──▸  paper        :  326 edges
   ✓ paper    ──belongs──▸ research_area:  334 edges
   ✓ advisor  ──expert──▸  research_area:  104 edges
   ✓ s